# Tutoriel K-ABENA — ML classique (niveau 1 : notebook débutant)
**Familles couvertes** : régression logistique, SVM linéaire, softmax multiclasse (scikit-learn).

**L'idée en une phrase** : à chaque époque, on met de côté une partie des exemples déjà bien appris,
et chaque exemple conservé *vote au nom de ceux qu'on a dispensés* (poids Horvitz-Thompson) —
même modèle, ~28 % de calcul en moins.

**La promesse 2 lignes** : `kb = Kabena()` puis `active, w = kb.select(losses)`. C'est tout.

In [ ]:
# Installation (une fois) :
# pip install kabena
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

D = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(D.data, D.target, test_size=0.25, random_state=0, stratify=D.target)
sc = StandardScaler().fit(Xtr)
Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
print("train:", Xtr.shape, "| test:", Xte.shape)

In [ ]:
# Régression logistique "maison" (30 lignes, pour tout voir) + K-ABENA en 2 lignes
from kabena import Kabena

def sigmoid(z): return 1/(1+np.exp(-np.clip(z, -30, 30)))
def per_sample_loss(w, X, y):
    p = np.clip(sigmoid(X @ w), 1e-9, 1-1e-9)
    return -(y*np.log(p) + (1-y)*np.log(1-p))
def grad(w, X, y, sw):
    p = sigmoid(X @ w)
    return X.T @ ((p - y) * sw) / sw.sum()

kb = Kabena(seed=0)                      # <= LIGNE 1  (défauts : v3, N=0.3)
w = np.zeros(Xtr.shape[1])
for epoch in range(60):
    losses = per_sample_loss(w, Xtr, ytr)
    active, sw = kb.select(losses)       # <= LIGNE 2
    w = w - 0.5 * grad(w, Xtr[active], ytr[active], sw[active])

acc = ((sigmoid(Xte @ w) > 0.5).astype(int) == yte).mean()
print(f"accuracy test = {acc:.4f}   |   calcul économisé = {kb.last_gain_*100:.1f}%")

## À vous de jouer
1. Remplacez `Kabena(seed=0)` par `Kabena(strategy='v2', seed=0)` : le **changement est transparent**
   (mêmes 2 lignes, même signature) — comparez l'accuracy.
2. Essayez `N=0.5` : plus d'exclusion, plus d'économie — l'accuracy tient-elle ?
3. SVM et softmax : mêmes 2 lignes, seule la fonction de perte change
   (voir `niveau2_script.py` pour les trois familles complètes).